In [1]:
import torch
import torch.nn as nn
from torch.optim import Adam
import matplotlib.pyplot as plt
import os
from tqdm import tqdm
import numpy as np
import shutil
import os

In [2]:
def visualize_and_save(model, val_loader, epoch, save_dir, device):
    model.eval()
    mean = torch.tensor([0.485, 0.456, 0.406]).view(1,3,1,1).to(device)
    std = torch.tensor([0.229, 0.224, 0.225]).view(1,3,1,1).to(device)

    batch = next(iter(val_loader))
    rgb = batch['rgb'][:2].to(device)
    gt = batch['ground_truth'][:2].to(device)

    with torch.no_grad():
        rectified, flow = model(rgb)

    fig, axes = plt.subplots(2, 3, figsize=(12, 8))
    for i in range(2):
        inp = (rgb[i:i+1] * std + mean).squeeze(0).permute(1,2,0).cpu().numpy().clip(0,1)
        rec = (rectified[i:i+1] * std + mean).squeeze(0).permute(1,2,0).cpu().numpy().clip(0,1)
        g = (gt[i:i+1] * std + mean).squeeze(0).permute(1,2,0).cpu().numpy().clip(0,1)

        axes[i,0].imshow(inp);  axes[i,0].set_title('Input');      axes[i,0].axis('off')
        axes[i,1].imshow(rec);  axes[i,1].set_title('Rectified');  axes[i,1].axis('off')
        axes[i,2].imshow(g);    axes[i,2].set_title('Ground Truth'); axes[i,2].axis('off')

    plt.suptitle(f'Epoch {epoch}')
    plt.tight_layout()
    save_path = os.path.join(save_dir, f'viz_epoch_{epoch:03d}.png')
    plt.savefig(save_path, dpi=100)
    plt.close()
    print(f"Saved visualization to {save_path}")
    model.train()


In [3]:
def visualize_and_save_uv(model, val_loader, epoch, save_dir, device):
    model.eval()

    batch = next(iter(val_loader))
    rgb = batch['rgb'][:2].to(device)
    gt_uv = batch['uv'][:2].to(device)

    with torch.no_grad():
        pred_uv = model(rgb)

    save_path = os.path.join(save_dir, f'uv_epoch_{epoch:03d}.pt')
    torch.save({
        'rgb': batch['rgb'][:2],
        'pred_uv': pred_uv.cpu(),
        'gt_uv': gt_uv.cpu(),
    }, save_path)
    print(f"Saved UV tensors to {save_path}")
    model.train()

In [4]:
def trainloop(model, criterion, optimizer, train_loader, val_loader, num_epochs, checkpoint_path, model_path, viz_dir):
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f"Using device: {device}")

    best_val_loss = float('inf')
    start_epoch = 0
    train_losses = []
    val_losses = []

    if viz_dir:
        os.makedirs(viz_dir, exist_ok=True)

    if os.path.exists(checkpoint_path):
        print("Resuming from checkpoint...")
        checkpoint = torch.load(checkpoint_path)
        model.load_state_dict(checkpoint['model_state'])
        optimizer.load_state_dict(checkpoint['optimizer_state'])
        start_epoch = checkpoint['epoch']
        train_losses = checkpoint['train_losses']
        val_losses = checkpoint['val_losses']
        best_val_loss = checkpoint['best_val_loss']
        print(f"Resumed from epoch {start_epoch}")

    for epoch in range(start_epoch, num_epochs):
        print(f"\n{'='*50}")
        print(f"Epoch {epoch+1}/{num_epochs}")
        print(f"{'='*50}")

        train_loss = train_one_epoch(model, train_loader, criterion, optimizer, device, epoch)
        print(f"Train Loss: {train_loss:.4f}")

        val_loss = validate(model, val_loader, criterion, device)
        print(f"Val Loss: {val_loss:.4f}")

        train_losses.append(train_loss)
        val_losses.append(val_loss)

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            torch.save(model.state_dict(), 'best_model.pth')
            shutil.copy('best_model.pth', model_path)
            print(f"Saved best model with val loss: {val_loss:.4f}")

        torch.save({
            'epoch': epoch + 1,
            'model_state': model.state_dict(),
            'optimizer_state': optimizer.state_dict(),
            'train_losses': train_losses,
            'val_losses': val_losses,
            'best_val_loss': best_val_loss,
        }, checkpoint_path)

        # if viz_dir and ((epoch + 1) % 5 == 0 or epoch == start_epoch):
        visualize_and_save_uv(model, val_loader, epoch + 1, viz_dir, device)

    print("\nTraining complete!")
    print(f"Best validation loss: {best_val_loss:.4f}")

In [ ]:
# lower the learning rate for the next epochs
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = DocumentReconstructionModel(model_type='m3').to(device)
criterion = nn.L1Loss()
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-5, weight_decay=0.01)
trainloop(
    model, criterion, optimizer,
    train_loader, val_loader,
    num_epochs=50,
    checkpoint_path=CHECKPOINT_PATH_M1,
    model_path=MODEL_PATH_M1,
    viz_dir=VIZ_PATH_M1
)

In [ ]:
data = torch.load('/content/drive/MyDrive/tufts/cs 132 computer vision/trainv3/viz_m1/uv_epoch_043.pt')

pred = data['pred_uv'][1].numpy()
gt = data['gt_uv'][1].numpy()

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

p_img = np.zeros((pred.shape[1], pred.shape[2], 3))
p_img[:,:,0] = (pred[0] + 1) / 2
p_img[:,:,1] = (pred[1] + 1) / 2

g_img = np.zeros((gt.shape[1], gt.shape[2], 3))
g_img[:,:,0] = (gt[0] + 1) / 2
g_img[:,:,1] = (gt[1] + 1) / 2

inp = data['rgb'][1]
mean = torch.tensor([0.485, 0.456, 0.406]).view(3,1,1)
std = torch.tensor([0.229, 0.224, 0.225]).view(3,1,1)
inp = (inp * std + mean).permute(1,2,0).numpy().clip(0,1)

axes[0].imshow(inp);             axes[0].set_title('Input')
axes[1].imshow(p_img.clip(0,1)); axes[1].set_title('Predicted UV')
axes[2].imshow(g_img.clip(0,1)); axes[2].set_title('GT UV')
plt.show()

In [ ]:
model = DocumentReconstructionModel(model_type='m3').to(device)
model.load_state_dict(torch.load(MODEL_PATH_M1))
model.eval()

batch = next(iter(val_loader))
rgb = batch['rgb'][:4].to(device)
gt_uv = batch['uv'][:4].to(device)

with torch.no_grad():
    pred_uv = model(rgb)

mean = torch.tensor([0.485, 0.456, 0.406]).view(1,3,1,1).to(device)
std = torch.tensor([0.229, 0.224, 0.225]).view(1,3,1,1).to(device)

fig, axes = plt.subplots(4, 3, figsize=(12, 16))
for i in range(4):
    inp = (rgb[i:i+1] * std + mean).squeeze(0).permute(1,2,0).cpu().numpy().clip(0,1)

    p = pred_uv[i].cpu().numpy()
    g = gt_uv[i].cpu().numpy()

    p_img = np.zeros((p.shape[1], p.shape[2], 3))
    p_img[:,:,0] = (p[0] + 1) / 2
    p_img[:,:,1] = (p[1] + 1) / 2

    g_img = np.zeros((g.shape[1], g.shape[2], 3))
    g_img[:,:,0] = (g[0] + 1) / 2
    g_img[:,:,1] = (g[1] + 1) / 2

    axes[i,0].imshow(inp);             axes[i,0].set_title('Input');        axes[i,0].axis('off')
    axes[i,1].imshow(p_img.clip(0,1)); axes[i,1].set_title('Predicted UV'); axes[i,1].axis('off')
    axes[i,2].imshow(g_img.clip(0,1)); axes[i,2].set_title('GT UV');        axes[i,2].axis('off')

plt.tight_layout()
plt.show()